# Fine-tune both models on new real-cage labels (YOLOv8n + YOLO11n)
Loads the two checkpoints you already trained (`yolov8n_pretrain` and `yolo11n_pretrain`) from Drive, fine-tunes each on `for_yolo_tune.zip`, and validates both on the **same** held-out test split for a fair comparison.

> No need to re-run pretraining. Set **Runtime → T4 GPU** first.


## 1. Setup


In [ ]:
!pip install -q ultralytics roboflow
import ultralytics, torch
ultralytics.checks()
print("CUDA:", torch.cuda.is_available())

Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)
CUDA: True


## 2. Mount Drive, locate the zip & the pretrained weights
Put `for_yolo_tune.zip` in your Drive root (`MyDrive/`). If it's still on your laptop, the cell will prompt an upload instead.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT_DIR = "/content/drive/MyDrive/mouse_detect"   # where your runs live
IMGSZ       = 640

# pretrained checkpoints from your two training runs
PRETRAINED = {
    "yolov8n": f"{PROJECT_DIR}/yolov8n_pretrain/weights/best.pt",
    "yolo11n": f"{PROJECT_DIR}/yolo11n_pretrain/weights/best.pt",
}
for name, p in PRETRAINED.items():
    print(f"{name:8s} weights found: {os.path.exists(p)}  ->  {p}")

# locate the new annotated zip (Drive root, /content, or upload)
CANDIDATES = ["/content/drive/MyDrive/for_yolo_tune.zip", "/content/for_yolo_tune.zip"]
FT_ZIP = next((p for p in CANDIDATES if os.path.exists(p)), None)
if FT_ZIP is None:
    print("Zip not found on Drive/content — upload it now:")
    from google.colab import files
    up = files.upload()                       # pick for_yolo_tune.zip
    FT_ZIP = "/content/" + list(up.keys())[0]
print("Using zip:", FT_ZIP)

Mounted at /content/drive
yolov8n  weights found: True  ->  /content/drive/MyDrive/mouse_detect/yolov8n_pretrain/weights/best.pt
yolo11n  weights found: True  ->  /content/drive/MyDrive/mouse_detect/yolo11n_pretrain/weights/best.pt
Using zip: /content/drive/MyDrive/for_yolo_tune.zip


## 3. Build the fine-tune dataset
Handles either case automatically: if your Roboflow export already has `train/valid/test`, those splits are kept; if it only has a `train/` folder, a contiguous split is made (no random shuffle, to limit near-duplicate frame leakage). All images are forced to grayscale to match the IR deployment camera and the pretraining pipeline.

> **Leakage note:** if you let Roboflow auto-split randomly and your frames are near-duplicates, the test mAP will be optimistic. A split by *recording session* is the honest version — worth doing before you trust the final number for deployment.


In [ ]:
import zipfile, glob, shutil, cv2, yaml

FT_RAW = "/content/ft_raw"
FT_DS  = "/content/real_cage_dataset"      # the fine-tune cell points here
for d in (FT_RAW, FT_DS):
    if os.path.isdir(d): shutil.rmtree(d)

with zipfile.ZipFile(FT_ZIP) as z:
    z.extractall(FT_RAW)

# carry class names over from the export's data.yaml (fallback: single 'mouse')
src_yaml = glob.glob(f"{FT_RAW}/**/data.yaml", recursive=True)
names, nc = ["mouse"], 1
if src_yaml:
    y = yaml.safe_load(open(src_yaml[0]))
    names = y.get("names", names); nc = y.get("nc", len(names))
print("classes:", names)

def imgs_in(split):
    return sorted(set(glob.glob(f"{FT_RAW}/**/{split}/images/*.*", recursive=True)))

def lbl_dir_for(img_path):
    return img_path.replace("/images/", "/labels/").rsplit("/", 1)[0]

have = {s: imgs_in(s) for s in ["train", "valid", "test"]}
print({s: len(v) for s, v in have.items()})

def place(file_list, split):
    os.makedirs(f"{FT_DS}/{split}/images", exist_ok=True)
    os.makedirs(f"{FT_DS}/{split}/labels", exist_ok=True)
    for img in file_list:
        stem = os.path.splitext(os.path.basename(img))[0]
        lbl  = os.path.join(lbl_dir_for(img), stem + ".txt")
        shutil.copy(img, f"{FT_DS}/{split}/images/")
        if os.path.exists(lbl):
            shutil.copy(lbl, f"{FT_DS}/{split}/labels/")
        else:                                      # empty label = background frame (valid)
            open(f"{FT_DS}/{split}/labels/{stem}.txt", "w").close()

if have["valid"] and have["test"]:
    print("Using the train/valid/test splits from the export as-is.")
    for s in ["train", "valid", "test"]:
        place(have[s], s)
else:
    all_imgs = have["train"] or sorted(
        set(glob.glob(f"{FT_RAW}/**/images/*.*", recursive=True)))
    n = len(all_imgs)
    n_test  = max(8,  round(n * 0.07))
    n_valid = max(15, round(n * 0.13))
    n_test  = min(n_test,  max(1, n - 2))
    n_valid = min(n_valid, max(1, n - n_test - 1))
    place(all_imgs[:n_test],                       "test")
    place(all_imgs[n_test:n_test + n_valid],       "valid")
    place(all_imgs[n_test + n_valid:],             "train")
    print(f"Contiguous split -> train {n - n_test - n_valid} | "
          f"valid {n_valid} | test {n_test}")

# clean data.yaml with absolute paths
data_yaml = {"path": FT_DS, "train": "train/images", "val": "valid/images",
             "test": "test/images", "nc": nc, "names": names}
REAL_DATA_YAML = f"{FT_DS}/real_cage_data.yaml"
yaml.dump(data_yaml, open(REAL_DATA_YAML, "w"))

# grayscale every split (match IR pipeline)
for split in ["train", "valid", "test"]:
    for p in glob.glob(f"{FT_DS}/{split}/images/*.*"):
        im = cv2.imread(p)
        if im is not None:
            g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
            cv2.imwrite(p, cv2.cvtColor(g, cv2.COLOR_GRAY2BGR))

for s in ["train", "valid", "test"]:
    print(s, len(glob.glob(f"{FT_DS}/{s}/images/*.*")), "images")
print("Dataset ready ->", REAL_DATA_YAML)

classes: ['Mouse']
{'train': 141, 'valid': 40, 'test': 20}
Using the train/valid/test splits from the export as-is.
train 141 images
valid 40 images
test 20 images
Dataset ready -> /content/real_cage_dataset/real_cage_data.yaml


## 4. Fine-tune both models & compare
Each model starts from its own pretrained `best.pt`, trains with a low LR and a frozen backbone (`freeze=10` — set to `0` if it underfits), then is validated on the identical test split. Interruption-safe: numbered checkpoints go to Drive and the cell auto-resumes after a disconnect.

Fine-tuned weights land at `mouse_detect/<model>_realcage_ft/weights/best.pt`.


In [ ]:
from ultralytics import YOLO
import pandas as pd, shutil

FT_RUNS = {"yolov8n": "yolov8n_realcage_ft", "yolo11n": "yolo11n_realcage_ft"}
EPOCHS  = 60
FORCE_RETRAIN = True   # wipe any prior run with the same name and train clean

# NOTE: no resume logic here on purpose. Fine-tuning is ~3 min; a fragile resume
# from a finalized/stale checkpoint can silently fall back to COCO defaults.
# We always train fresh from the pretrained weights with data= passed explicitly,
# so the run can never touch coco8.

def finetune(pretrained, ft_name):
    run_dir = f"{PROJECT_DIR}/{ft_name}"
    best    = f"{run_dir}/weights/best.pt"

    if FORCE_RETRAIN and os.path.isdir(run_dir):
        print(f"[{ft_name}] removing stale run dir -> {run_dir}")
        shutil.rmtree(run_dir)

    if os.path.exists(best) and not FORCE_RETRAIN:
        print(f"[{ft_name}] reusing existing fine-tuned weights")
    else:
        print(f"[{ft_name}] fine-tuning from {pretrained}")
        YOLO(pretrained).train(
            data=REAL_DATA_YAML, epochs=EPOCHS, patience=20, save_period=5,
            imgsz=IMGSZ, batch=16, lr0=0.001, cos_lr=True, freeze=10,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.2,
            degrees=0.0, translate=0.05, scale=0.2, fliplr=0.5, flipud=0.0,
            mosaic=0.0, close_mosaic=10,
            project=PROJECT_DIR, name=ft_name, exist_ok=True,
        )

    # sanity check: the fine-tuned model must be single-class ('mouse'), not COCO
    names = YOLO(best).names
    if len(names) != 1:
        print(f"   !! WARNING: {ft_name} has {len(names)} classes {names} "
              f"-- expected 1. Something fell back to COCO; investigate.")

    m = YOLO(best).val(data=REAL_DATA_YAML, split="test", imgsz=IMGSZ)
    return {"model": ft_name.replace("_realcage_ft", ""),
            "mAP50": round(m.box.map50, 4), "mAP50-95": round(m.box.map, 4),
            "precision": round(m.box.mp, 4), "recall": round(m.box.mr, 4)}

rows = []
for name, ft_name in FT_RUNS.items():
    pre = PRETRAINED[name]
    if not os.path.exists(pre):
        print(f"!! skipping {name}: pretrained weights missing at {pre}")
        continue
    rows.append(finetune(pre, ft_name))

print("\n=== Real-cage test-split comparison ===")
print(pd.DataFrame(rows).to_string(index=False))

[yolov8n_realcage_ft] removing stale run dir -> /content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft
[yolov8n_realcage_ft] fine-tuning from /content/drive/MyDrive/mouse_detect/yolov8n_pretrain/weights/best.pt
Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/real_cage_dataset/real_cage_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.

## 5. (Optional) Eyeball the fine-tuned detectors on raw frames
Re-runs the fine-tuned weights over `cage_frames.zip` so you can see the before/after jump in detection rate.


In [ ]:
import random
FRAMES_DIR, ZIP_PATH = "/content/cage_frames", "/content/drive/MyDrive/cage_frames.zip"
if os.path.exists(ZIP_PATH):
    if not os.path.isdir(FRAMES_DIR) or not os.listdir(FRAMES_DIR):
        with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(FRAMES_DIR)
    frames = sorted(f for ext in ("*.jpg","*.jpeg","*.png","*.bmp")
                    for f in glob.glob(f"{FRAMES_DIR}/**/{ext}", recursive=True))
    sample = random.sample(frames, min(30, len(frames)))
    for name, ft_name in FT_RUNS.items():
        best = f"{PROJECT_DIR}/{ft_name}/weights/best.pt"
        if not os.path.exists(best): continue
        r = YOLO(best).predict(sample, conf=0.25, imgsz=IMGSZ, save=True,
                               project=PROJECT_DIR, name=f"{name}_ft_check", exist_ok=True, verbose=False)
        hits = sum(1 for x in r if len(x.boxes))
        print(f"{name}: {hits}/{len(sample)} sampled frames detected -> "
              f"{PROJECT_DIR}/{name}_ft_check/")
else:
    print("cage_frames.zip not found on Drive — skip or fix the path.")

Results saved to /content/drive/MyDrive/mouse_detect/yolov8n_ft_check
yolov8n: 30/30 sampled frames detected -> /content/drive/MyDrive/mouse_detect/yolov8n_ft_check/
Results saved to /content/drive/MyDrive/mouse_detect/yolo11n_ft_check
yolo11n: 30/30 sampled frames detected -> /content/drive/MyDrive/mouse_detect/yolo11n_ft_check/


In [ ]:
from ultralytics import YOLO
from google.colab import files
import os, shutil

PROJECT_DIR = "/content/drive/MyDrive/mouse_detect"
FT_BEST = f"{PROJECT_DIR}/yolov8n_realcage_ft/weights/best.pt"
assert os.path.exists(FT_BEST), f"Not found: {FT_BEST}"

EXPORT_DIR = f"{PROJECT_DIR}/deploy"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) Simple version — native PyTorch .pt
pt_out = f"{EXPORT_DIR}/yolov8n_mouse.pt"
shutil.copy(FT_BEST, pt_out)

# 2) ONNX — portable, runs on CPU via onnxruntime / OpenVINO / OpenCV DNN
model = YOLO(FT_BEST)
onnx_tmp = model.export(format="onnx", imgsz=640, simplify=True)  # add opset=12 if your runtime is older
onnx_out = f"{EXPORT_DIR}/yolov8n_mouse.onnx"
shutil.copy(onnx_tmp, onnx_out)

print("Saved to Drive:")
print(" ", pt_out,   "(", round(os.path.getsize(pt_out)/1e6, 1),  "MB )")
print(" ", onnx_out, "(", round(os.path.getsize(onnx_out)/1e6, 1), "MB )")

# download both to your machine
files.download(pt_out)
files.download(onnx_out)

Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 769ms
Prepared 4 packages in 2.77s
Installed 4 packages in 421ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 4.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 ops

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import YOLO
from google.colab import files
import os, shutil

PROJECT_DIR = "/content/drive/MyDrive/mouse_detect"
EXPORT_DIR  = f"{PROJECT_DIR}/deploy"
os.makedirs(EXPORT_DIR, exist_ok=True)

MODELS = {
    "yolov8n_mouse": f"{PROJECT_DIR}/yolov8n_realcage_ft/weights/best.pt",
    "yolo11n_mouse": f"{PROJECT_DIR}/yolo11n_realcage_ft/weights/best.pt",
}

to_download = []
for name, src in MODELS.items():
    if not os.path.exists(src):
        print(f"!! skipping {name}: not found at {src}")
        continue
    pt_out = f"{EXPORT_DIR}/{name}.pt"            # simple PyTorch version
    shutil.copy(src, pt_out)
    onnx_tmp = YOLO(src).export(format="onnx", imgsz=640, simplify=True)
    onnx_out = f"{EXPORT_DIR}/{name}.onnx"        # portable ONNX version
    shutil.copy(onnx_tmp, onnx_out)
    to_download += [pt_out, onnx_out]
    print(f"{name}: .pt {round(os.path.getsize(pt_out)/1e6,1)}MB | "
          f".onnx {round(os.path.getsize(onnx_out)/1e6,1)}MB")

for f in to_download:
    files.download(f)

Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.5s, saved as '/content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft/weights/best.onnx' (11.7 MB)

Export complete (3.3s)
Results saved to /content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/mouse_detect/yolov8n_realcage_ft/weights/best.onnx imgsz=640 data=/content/real_cage_dataset/real_cage_data.yaml  
Visualize:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>